# Notebook 4: Correcting Langevin Bias

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FloppingCode/modified-langevin-score-matching/blob/main/notebooks/04_modified_langevin.ipynb)

**Goal:** Explore the bias in annealed Langevin dynamics and the role of a correction term. Compare standard vs modified Langevin for both learned and analytical scores.

This notebook loads the pretrained NCSN from Notebook 03 (no training required) and focuses entirely on the sampling side.

## Setup

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    if os.path.exists("modified-langevin-score-matching"):
        !cd modified-langevin-score-matching && git pull
    else:
        !git clone https://github.com/FloppingCode/modified-langevin-score-matching.git
    sys.path.insert(0, "modified-langevin-score-matching")
else:
    sys.path.insert(0, "..")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from dsm import (
    make_dataset,
    GeometricNoiseSchedule,
    annealed_langevin_dynamics,
    modified_langevin_dynamics,
    make_8gaussians_analytical_score,
    load_checkpoint,
)
from dsm.visualization import plot_samples, animate_sampling, display_animation

## Configuration

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Sampling config
N_GENERATED = 2000
STEPS_PER_SIGMA = 100
STEP_SIZE_FACTOR = 5e-5
SAVE_EVERY = 5  # ~200 frames

## Load Pretrained NCSN

This checkpoint was saved by Notebook 03. If it does not exist, run Notebook 03 first.

In [ ]:
CHECKPOINT_DIR = "checkpoints" if "google.colab" not in sys.modules else "modified-langevin-score-matching/checkpoints"
checkpoint_path = os.path.join(CHECKPOINT_DIR, "8gaussians_trained.pt")

assert os.path.exists(checkpoint_path), (
    f"Checkpoint not found at {checkpoint_path}. "
    "Please run Notebook 03 first to train and save the NCSN model."
)

model, noise_schedule, config, history = load_checkpoint(checkpoint_path, device=DEVICE)
model.eval()

print(f"Loaded model config: {config}")
print(f"Noise levels: {noise_schedule.sigmas.tolist()}")
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## Create Dataset & Analytical Model (for comparison)

In [ ]:
dataset = make_dataset("8gaussians", n_samples=10_000)
data = dataset.tensors[0]
analytical_model = make_8gaussians_analytical_score().to(DEVICE)

print(f"Dataset shape: {data.shape}")

## The Bias Problem

Annealed Langevin dynamics samples using the score of the **noised** distribution:
$$\nabla_x \log p_\sigma(x) \quad \text{where} \quad p_\sigma(x) = \int p(y) \mathcal{N}(x; y, \sigma^2 I) \, dy$$

But we want samples from the **clean** distribution $p_0(x) = p(x)$.

Even with infinitely many Langevin steps at each noise level, the final samples come from $p_{\sigma_L}(x)$ (the smallest noise level), not $p_0(x)$. When $\sigma_L > 0$, there is a residual bias -- the samples are slightly over-smoothed.

The **correction term** addresses this gap by adjusting the Langevin update:
$$x_{t+1} = x_t + \frac{\alpha}{2} s_\theta(x_t, \sigma) + c(x_t, s_\theta, \sigma, \alpha) + \sqrt{\alpha}\, z_t$$

where $c(\cdot)$ compensates for the difference between $\nabla_x \log p_\sigma$ and $\nabla_x \log p_0$.

## Standard Annealed Langevin with Learned NCSN

In [ ]:
std_learned_samples, std_learned_traj = annealed_langevin_dynamics(
    model,
    noise_schedule,
    n_samples=N_GENERATED,
    data_dim=2,
    steps_per_sigma=STEPS_PER_SIGMA,
    step_size_factor=STEP_SIZE_FACTOR,
    device=DEVICE,
    return_trajectories=True,
    save_every=SAVE_EVERY,
)
print(f"Standard learned samples: {std_learned_samples.shape}, frames: {std_learned_traj.shape[0]}")

plot_samples(data, std_learned_samples.cpu(), title="Standard Annealed Langevin (Learned NCSN)")
plt.show()

In [ ]:
anim_std_learned = animate_sampling(
    std_learned_traj,
    real_data=data,
    n_particles=200,
    interval=50,
    title="Standard Annealed (Learned NCSN)",
    trail_length=10,
)
display_animation(anim_std_learned)

## Standard Annealed Langevin with Analytical Score

In [ ]:
std_analytical_samples, std_analytical_traj = annealed_langevin_dynamics(
    analytical_model,
    noise_schedule,
    n_samples=N_GENERATED,
    data_dim=2,
    steps_per_sigma=STEPS_PER_SIGMA,
    step_size_factor=STEP_SIZE_FACTOR,
    device=DEVICE,
    return_trajectories=True,
    save_every=SAVE_EVERY,
)
print(f"Standard analytical samples: {std_analytical_samples.shape}, frames: {std_analytical_traj.shape[0]}")

plot_samples(data, std_analytical_samples.cpu(), title="Standard Annealed Langevin (Analytical Score)")
plt.show()

In [ ]:
anim_std_analytical = animate_sampling(
    std_analytical_traj,
    real_data=data,
    n_particles=200,
    interval=50,
    title="Standard Annealed (Analytical Score)",
    trail_length=10,
)
display_animation(anim_std_analytical)

## Define the Correction Function

The correction function receives the current state and returns an additive term. Implement your correction below.

With the zero placeholder, modified Langevin is identical to standard annealed Langevin.

In [ ]:
def correction_fn(x, score, sigma, alpha):
    """Correction term for modified Langevin dynamics.
    
    The score s(x, sigma) estimates nabla_x log p_sigma(x),
    but sampling targets nabla_x log p_0(x).
    This correction accounts for the gap.
    
    Args:
        x: (n_samples, data_dim) current particle positions
        score: (n_samples, data_dim) score estimate s(x, sigma)
        sigma: scalar tensor, current noise level
        alpha: scalar tensor, current step size
    
    Returns:
        correction: (n_samples, data_dim) additive correction
    """
    # TODO: Insert your correction formula
    return torch.zeros_like(x)

## Modified Langevin with Learned NCSN

In [ ]:
mod_learned_samples, mod_learned_traj = modified_langevin_dynamics(
    model,
    noise_schedule,
    correction_fn=correction_fn,
    n_samples=N_GENERATED,
    data_dim=2,
    steps_per_sigma=STEPS_PER_SIGMA,
    step_size_factor=STEP_SIZE_FACTOR,
    device=DEVICE,
    return_trajectories=True,
    save_every=SAVE_EVERY,
)
print(f"Modified learned samples: {mod_learned_samples.shape}, frames: {mod_learned_traj.shape[0]}")

plot_samples(data, mod_learned_samples.cpu(), title="Modified Langevin (Learned NCSN)")
plt.show()

In [ ]:
anim_mod_learned = animate_sampling(
    mod_learned_traj,
    real_data=data,
    n_particles=200,
    interval=50,
    title="Modified Langevin (Learned NCSN)",
    trail_length=10,
)
display_animation(anim_mod_learned)

## Modified Langevin with Analytical Score

In [ ]:
mod_analytical_samples, mod_analytical_traj = modified_langevin_dynamics(
    analytical_model,
    noise_schedule,
    correction_fn=correction_fn,
    n_samples=N_GENERATED,
    data_dim=2,
    steps_per_sigma=STEPS_PER_SIGMA,
    step_size_factor=STEP_SIZE_FACTOR,
    device=DEVICE,
    return_trajectories=True,
    save_every=SAVE_EVERY,
)
print(f"Modified analytical samples: {mod_analytical_samples.shape}, frames: {mod_analytical_traj.shape[0]}")

plot_samples(data, mod_analytical_samples.cpu(), title="Modified Langevin (Analytical Score)")
plt.show()

In [ ]:
anim_mod_analytical = animate_sampling(
    mod_analytical_traj,
    real_data=data,
    n_particles=200,
    interval=50,
    title="Modified Langevin (Analytical Score)",
    trail_length=10,
)
display_animation(anim_mod_analytical)

## 4-Panel Comparison

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

panels = [
    (std_analytical_samples, "Analytical\nStandard", "C2"),
    (mod_analytical_samples, "Analytical\nModified", "C4"),
    (std_learned_samples, "Learned\nStandard", "C1"),
    (mod_learned_samples, "Learned\nModified", "C3"),
]

for ax, (samples, title, color) in zip(axes, panels):
    s_np = samples.cpu().numpy()
    ax.scatter(s_np[:, 0], s_np[:, 1], s=1, alpha=0.5, color=color)
    ax.set_title(title, fontsize=12)
    ax.set_aspect("equal")
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

fig.suptitle("Standard vs Modified Langevin: Analytical vs Learned Score", fontsize=14)
fig.tight_layout()
plt.show()

## Quantitative Comparison

We compute the **mean distance to nearest real data point** for each sampling method. Lower values indicate better coverage of the true data distribution.

In [ ]:
def mean_nearest_distance(generated, real):
    """Mean L2 distance from each generated point to its nearest real point."""
    gen = generated.cpu()
    ref = real.cpu()
    # (N_gen, N_real)
    dists = torch.cdist(gen.unsqueeze(0), ref.unsqueeze(0)).squeeze(0)
    nearest = dists.min(dim=1).values
    return nearest.mean().item(), nearest.std().item()

methods = {
    "Analytical (Standard)": std_analytical_samples,
    "Analytical (Modified)": mod_analytical_samples,
    "Learned (Standard)": std_learned_samples,
    "Learned (Modified)": mod_learned_samples,
}

print(f"{'Method':<30s} {'Mean NN Dist':>14s} {'Std':>10s}")
print("-" * 56)
for name, samples in methods.items():
    mean_d, std_d = mean_nearest_distance(samples, data)
    print(f"{name:<30s} {mean_d:>14.6f} {std_d:>10.6f}")

**Note:** With the zero-placeholder correction, "Standard" and "Modified" should produce very similar metrics (any differences are due to random noise in Langevin dynamics). Once a non-trivial correction is implemented, the modified variants should show improvement.

## Summary

- **Standard annealed Langevin** samples from $p_{\sigma_L}(x)$, not $p_0(x)$. The residual smoothing at $\sigma_L > 0$ introduces bias.
- The **modified Langevin** framework adds a correction term to the update rule. With the right correction, it can reduce or eliminate this bias.
- With `correction_fn` returning zeros (placeholder), modified Langevin is equivalent to standard annealed Langevin.
- The **analytical score** isolates the sampling bias from score estimation error, letting us study each source of error independently.
- The quantitative metric (mean nearest-neighbor distance) provides a simple way to compare sample quality.

**To experiment:** Edit the `correction_fn` cell above with your correction formula and re-run the notebook to see the effect on both learned and analytical score sampling.